# Neon RAG benchmark matrix

This notebook runs Vidore with its three embedding-model choices and one optional chat model. Full ingestion is opt-in; the default is a deterministic offline smoke path.

Retrieval qrels measure whether relevant Vidore documents were retrieved (recall/MRR); answer metrics measure the generated answer separately. Correctness uses deterministic token-F1 against Vidore acceptable answers, with limited normalized exact match as a secondary deterministic check. Answer relevance is unavailable without a judge. Retrieved-passage citation coverage is distinct from claim-level citation completeness. Citation and groundedness checks are deterministic lexical/heuristic checks, not semantic entailment: they require parseable citations and associate each claim only with its cited passages.

In [ ]:
from neon_rag_benchmarks.config import BenchmarkConfig, MODELS
from neon_rag_benchmarks.pipeline import run_matrix

config = BenchmarkConfig.from_env()
SMOKE = True
DATASET = "vidore"  # Vidore is the supported answer-evaluation dataset
EMBEDDING_MODEL = "all"  # or nomic, minilm, mpnet
dataset_names = (DATASET,)
model_keys = None if EMBEDDING_MODEL == "all" else (EMBEDDING_MODEL,)
selected_count = len(dataset_names) * len(model_keys or tuple(MODELS))
print("controls:", DATASET, EMBEDDING_MODEL, config.gateway_model)

In [ ]:
# Live mode requires DATABASE_URL for retrieval; Databricks settings are optional for answer generation.
results = run_matrix(
    config,
    smoke=SMOKE,
    persist=config.persist_results,
    dataset_names=dataset_names,
    model_keys=model_keys,
)
print(f"{selected_count} selected combinations; {len(results)} chat-model records in this run")
print(results[0])

## Evaluation notes

The default controls select Vidore and all three local embedding models, so the run has three combinations. Native qrels are normalized retrieval annotations and retrieval relevance; generated-answer correctness, retrieved-passage citation coverage, claim-level citation completeness/validity, and lexical/heuristic claim grounding are answer evaluation. Canonical scopes are `complete`, `bounded`, or `skipped`; detail fields retain `synthetic`/`bounded_sample` explanations. `provenance_scope=complete` means all source corpus/query rows were retained. Dataset/model initialization timings are separate from per-request `rag_evaluation`. No answer-relevance score is fabricated when an evaluator/judge is not configured.